In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/train.parquet')
test = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/test.parquet')
train_input = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/train_input.parquet')
val_input = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/val_input.parquet')
val = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/val.parquet')

train_target = train.loc[train_input.index]['fare_amount']
val_target = val.loc[val_input.index]['fare_amount']

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

model_xgb = XGBRegressor(booster = 'dart',
                         n_jobs = -1,
                         n_estimators = 40,
                         max_depth = 5,
                         gamma = 0.4,
                         eta = 0.315,
                         random_state=123)
model_xgb.fit(train_input, train_target)
train_pred = model_xgb.predict(train_input)
val_pred = model_xgb.predict(val_input)
train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))
print(f'trian_rmse: {train_rmse}, val_rmse: {val_rmse}')

test_pred_xgb = model_xgb.predict(test)
submission_xgb = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/sample_submission.csv')
submission_xgb['fare_amount'] = test_pred_xgb
submission_xgb.to_csv('xgboost_sub.csv', index=None)